In [47]:
import numpy as np
from scipy import stats

np.random.seed(42)

def calc_confidence_intervals(sample, alpha=0.05):
    n = len(sample)
    mean_hat = np.mean(sample)
    var_hat = np.var(sample, ddof=1)
    std_hat = np.std(sample, ddof=1)
    
    u_crit = stats.norm.ppf(1 - alpha/2)
    margin_of_error = u_crit * (std_hat / np.sqrt(n))
    ci_mean = (mean_hat - margin_of_error, mean_hat + margin_of_error)
    
    e = stats.kurtosis(sample, fisher=True)
    U = u_crit * np.sqrt((e + 2) / n)
    
    lower_std = std_hat * (1 + U)**(-0.5)
    upper_std = std_hat * (1 - U)**(-0.5) if U < 1 else np.inf
    ci_var = (lower_std**2, upper_std**2)
    
    return mean_hat, var_hat, ci_mean, ci_var

def f_test(sample1, sample2, alpha=0.05):
    var1 = np.var(sample1, ddof=1)
    var2 = np.var(sample2, ddof=1)
    n1, n2 = len(sample1), len(sample2)
    
    if var1 >= var2:
        var_max, var_min = var1, var2
        df_num, df_den = n1 - 1, n2 - 1
        larger_name = "Выборка 1"
    else:
        var_max, var_min = var2, var1
        df_num, df_den = n2 - 1, n1 - 1
        larger_name = "Выборка 2"
    
    f_stat = var_max / var_min
    f_crit = stats.f.ppf(1 - alpha, df_num, df_den)
    condition = f_stat > f_crit
    decision = "Принимается (H0: Дисперсии равны)" if not condition else f"ОТКЛОНЯЕТСЯ (Дисперсия '{larger_name}' значимо больше)"
    
    return f_stat, f_crit, condition, decision, larger_name

n1, n2 = 20, 100
sample1 = np.random.normal(0, 1, n1)
sample2 = np.random.normal(0, 1, n2)

mean1, var1, ci_mean1, ci_var1 = calc_confidence_intervals(sample1)
mean2, var2, ci_mean2, ci_var2 = calc_confidence_intervals(sample2)

f_stat, f_crit, f_condition, f_decision, larger_name = f_test(sample1, sample2)

print("="*80)
print("РЕЗУЛЬТАТЫ ЛАБОРАТОРНОЙ РАБОТЫ №8")
print("="*80)

print("\nЗНАЧЕНИЯ ДОВЕРИТЕЛЬНЫХ ИНТЕРВАЛОВ И ТОЧЕЧНЫХ ОЦЕНОК")
print("-"*80)
print(f"{'Выборка':<20} {'m':>8} {'ДИ для m':<30} {'s²':>8} {'ДИ для s²':<25}")
print("-"*80)
print(f"{'Выборка 1 (n=20)':<20} {mean1:>8.3f} [{ci_mean1[0]:.3f}, {ci_mean1[1]:.3f}] {var1:>8.3f} [{ci_var1[0]:.3f}, {ci_var1[1]:.3f}]")
print(f"{'Выборка 2 (n=100)':<20} {mean2:>8.3f} [{ci_mean2[0]:.3f}, {ci_mean2[1]:.3f}] {var2:>8.3f} [{ci_var2[0]:.3f}, {ci_var2[1]:.3f}]")

print("\nРЕЗУЛЬТАТЫ ТЕСТА ФИШЕРА")
print("-"*50)
print(f"{'Показатель':<30} {'Значение':>20}")
print("-"*50)
print(f"{'В числителе':<30} {larger_name:>20} (дисперсия больше)")
print(f"{'F-статистика (набл.)':<30} {f_stat:>20.4f}")
print(f"{'F-критическое (прав.)':<30} {f_crit:>20.4f}")
print(f"{'Условие F > F_crit':<30} {f_condition:>20}")
print(f"{'Результат':<30} {f_decision:>20}")
print("="*80)

РЕЗУЛЬТАТЫ ЛАБОРАТОРНОЙ РАБОТЫ №8

ЗНАЧЕНИЯ ДОВЕРИТЕЛЬНЫХ ИНТЕРВАЛОВ И ТОЧЕЧНЫХ ОЦЕНОК
--------------------------------------------------------------------------------
Выборка                     m ДИ для m                             s² ДИ для s²                
--------------------------------------------------------------------------------
Выборка 1 (n=20)       -0.171 [-0.592, 0.249]    0.922 [0.603, 1.956]
Выборка 2 (n=100)      -0.061 [-0.241, 0.120]    0.850 [0.658, 1.202]

РЕЗУЛЬТАТЫ ТЕСТА ФИШЕРА
--------------------------------------------------
Показатель                                 Значение
--------------------------------------------------
В числителе                               Выборка 1 (дисперсия больше)
F-статистика (набл.)                         1.0840
F-критическое (прав.)                        1.6926
Условие F > F_crit                                0
Результат                      Принимается (H0: Дисперсии равны)
